In [ ]:
import numpy as np
import pandas as pd

In [ ]:
train = pd.read_csv("/kaggle/input/competitions/industrial-surface-defect-detection/public/train.csv")

In [ ]:
train.head()

In [ ]:
# rle_m = train['mask_rle'].notna()
# print(rle_m)
# # train['mask_rle'][0].split(" ")

In [ ]:
img = "/kaggle/input/competitions/industrial-surface-defect-detection/public/train/0ZUU8LH57Y.jpg"

## Q2

In [ ]:
def rle_decode(rle, h=128, w=800):
    # Empty mask_rle means "no defect" → return an all-zero mask
    if pd.isna(rle) or str(rle).strip() == "":
        return np.zeros((h, w), dtype=np.uint8)

    nums = np.array(str(rle).split(), dtype=int)   # "2 3 9 2" → [2, 3, 9, 2]
    starts = nums[0::2] - 1                         # [2, 9] → minus 1 → [1, 8]
    lengths = nums[1::2]                            # [3, 2]

    flat = np.zeros(h * w, dtype=np.uint8)          # one long line of zeros
    for s, l in zip(starts, lengths):
        flat[s:s + l] = 1                           # put 1s along each run

    return flat.reshape((h, w), order="F")          # fold into the grid, column by column

In [ ]:
#testing
print(rle_decode("2 3 9 2", h=4, w=4))

In [ ]:
# Q2: total defect pixels for image 0ZUU8LH57Y
rows = train[train["image_id"] == "0ZUU8LH57Y"]    # keep only this image's 4 rows

total = 0
for _, row in rows.iterrows():                      # go through the rows one by one
    mask = rle_decode(row["mask_rle"])              # RLE text → 128×800 mask
    count = int(mask.sum())                         # number of 1s = defect pixels
    print(row["defect_type"], mask.shape, count)
    total += count

print("Total defect pixels:", total)

## Q3

In [ ]:
# Q3: number of images containing each defect type
train["has_defect"] = train["mask_rle"].notna()
print(train[["image_id", "defect_type", "has_defect"]].head(4))

In [ ]:
positive = train[train["has_defect"]]      # keep only rows where a defect is present

counts = (positive.groupby("defect_type")["image_id"]
                  .nunique()
                  .sort_values(ascending=False))
print(counts)

print("Most common:", counts.index[0], counts.iloc[0])
print("Rarest    :", counts.index[-1], counts.iloc[-1])

## Q4 — Defect co-occurrence (4×4)
presence = one row per image, 1/0 per defect. presence.T @ presence counts, for each pair, the images where both are 1 (the diagonal = per-class counts from Q3).
Top pair: defect_c & defect_d, 705 images.

In [ ]:
# Q4: co-occurrence of defect types
presence = train.pivot(index="image_id",
                       columns="defect_type",
                       values="has_defect").astype(int)
print(presence.head(3))
print(presence.shape)

In [ ]:
co = presence.T @ presence
print(co)

In [ ]:
cols = list(co.columns)
best_pair, best_count = None, -1

for i in range(len(cols)):
    for j in range(i + 1, len(cols)):          # j > i → only the upper half, skips the diagonal
        if co.iloc[i, j] > best_count:
            best_count = co.iloc[i, j]
            best_pair = (cols[i], cols[j])

print("Top pair:", best_pair, best_count)